# [Scorio Trace](https://huggingface.co/datasets/harimo/scorio-trace)

This dataset contains 192,000 sampled reasoning traces: 20 model configs x 4 math benchmarks x 30 questions x 80 runs.

## Math benchmarks (tasks)

Every config has the same 4 splits (`aime24`, `aime25`, `brumo25`, `hmmt25_feb`), 2,400 rows each. The unit you actually work with is the candidate pool: the 80 traces of one (model, task, question), stored together and ordered by seed, so `pool[:n]` is a reproducible n-sample budget.

| config | rows | size | contents |
|---|---:|---:|---|
| `meta` (default) | 192,000 | 1.36 GiB | all 20 models, no per-token lists |
| 20 per-model configs | 9,600 each | 11.72 GiB total | adds the per-token lists |

## What is in a row

A row is one complete generation.

**Identity**

| column | type | notes |
|---|---|---|
| `task` | string | `aime24`, `aime25`, `brumo25`, `hmmt25_feb` |
| `model` | string | HF model id. gpt-oss configs share one id |
| `model_key` | string | config name. Group on this, not on `model` |
| `reasoning_level` | string | `low`, `medium` or `high` only for gpt-oss, null for the other 17 |
| `data_id` | int32 | question, 0 to 29 |
| `seed` | int32 | 1234 to 1313, 80 per question |

**Input**

| column | type | notes |
|---|---|---|
| `prompt` | string | the problem statement |
| `trigger` | string | the instruction wrapped around it |
| `has_trigger` | bool | 1.0 for every model except gpt-oss |
| `sampling` | struct | strategy, temperature, top_p, n, max_tokens, skip_special_tokens, seed |
| `num_prompt_tokens` | int32 | |

**Output**

| column | type | notes |
|---|---|---|
| `text` | string | the full generation |
| `finish_reason` | string | `stop`, or `length` if it hit the 32,768 token cap |
| `num_completion_tokens` | int32 | |

**Grading (binary correctness)**

| column | type | notes |
|---|---|---|
| `ground_truth` | string | a Python `repr()` string. Use `ast.literal_eval` to parse it |
| `ground_truth_accepted` | list of string | already parsed and de-duplicated, preferred over `ground_truth` |
| `extracted_answer` | string | what was pulled out of the `\boxed{}` span |
| `has_box` | int8 | whether the trace produced a `\boxed{}` at all |
| `is_correct` | int8 | the grade |

**Grading (reward models)**, CompassVerifier 3B and 7B:

| column | type | notes |
|---|---|---|
| `cv3b_label`, `cv7b_label` | string | verifier label, `A` means judged correct |
| `cv3b_prob`, `cv7b_prob` | double | probability of that label |
| `cv3b_ctx_A/B/C`, `cv7b_ctx_A/B/C` | double | contextual probabilities |

Score as P(correct): `cv*_prob` if `cv*_label == "A"`, else `1 - cv*_prob`.

**Token statistics**, all inside the `tokens` struct:

| field | `meta` | per-model | length |
|---|---|---|---|
| `prompt_sum_logprob`, `prompt_avg_logprob`, `prompt_ppl` | yes | yes | scalar |
| `completion_sum_logprob`, `completion_avg_logprob`, `completion_ppl` | yes | yes | scalar |
| `prompt_token_list`, `prompt_logprob_list`, `prompt_rank_list` | no | yes | `num_prompt_tokens` |
| `completion_token_list`, `completion_logprob_list`, `completion_rank_list` | no | yes | `num_completion_tokens` |

The list fields hold one entry per token of the trace. The scalars are just their
aggregates, which is why `meta` is 8x smaller. Only the realized token's logprob
and rank were stored, there is no top-k, so perplexity style metrics reproduce but
entropy style ones do not. Prompt position 0 is null in all three prompt-side lists.


## Exceptions

Two columns exist only to flag edge cases.

| column | type | true for |
|---|---|---|
| `answer_is_set`, `extracted_answer_is_list` | bool | the 1,600 rows of `brumo25` q22, the one set-valued question. False everywhere else |
| `prompt_logprob_sentinel` | bool | all 9,600 rows of `NVIDIA-Nemotron-Nano-9B-v2`, whose prompt logprobs contain infinities. No other model |



## Loading the dataset



In [1]:
from collections import Counter
from datasets import load_dataset, get_dataset_config_names, get_dataset_split_names

repo_name = "harimo/scorio-trace"

models = get_dataset_config_names(repo_name)

tasks = get_dataset_split_names(repo_name, "meta")   # ['aime24', 'aime25', 'brumo25', 'hmmt25_feb']


print(f"Models: {models}")
print(f"Tasks: {tasks}")

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

...

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Models: ['meta', 'AceReason-Nemotron-1.1-7B', 'Bespoke-Stratos-7B', 'DeepSeek-R1-Distill-Qwen-1.5B', 'EXAONE-4.0-1.2B', 'FuseO1-DeepSeekR1-QwQ-SkyT1-Flash-32B-Preview', 'LIMO-v2', 'Light-R1-14B-DS', 'NVIDIA-Nemotron-Nano-9B-v2', 'OpenR1-Distill-7B', 'OpenReasoning-Nemotron-1.5B', 'OpenThinker2-32B', 'OpenThinker3-1.5B', 'Phi-4-reasoning', 'Phi-4-reasoning-plus', 'Qwen3-30B-A3B-Thinking-2507', 'Qwen3-4B-Thinking-2507', 'Sky-T1-32B-Flash', 'gpt-oss-20b_high', 'gpt-oss-20b_low', 'gpt-oss-20b_medium']
Tasks: ['aime24', 'aime25', 'brumo25', 'hmmt25_feb']


## All data, 20 models x 4 tasks (the `meta` config)

`load_dataset(repo_name)` without a config name gives you the default config, `meta`.
That is all 20 models in 4 splits of 48,000 rows, everything except the per-token
lists. 1.36 GiB packed, about 7.7 GiB once it is in Arrow.

Use this tier for accuracy, ranking, pass@k and reward model work. Switch to a
per-model config when you need per-token logprobs and ranks.


In [2]:
ds = load_dataset(repo_name)  # load all datasets for all models
# 20 (models) x 4 (datasets) x 30 (questions) x 80 (runs) = 192,000 runs

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

...

meta/hmmt25_feb/gpt-oss-20b_medium.parqu(…):   0%|          | 0.00/20.0M [00:00<?, ?B/s]

Generating aime24 split:   0%|          | 0/48000 [00:00<?, ? examples/s]

Generating aime25 split:   0%|          | 0/48000 [00:00<?, ? examples/s]

Generating brumo25 split:   0%|          | 0/48000 [00:00<?, ? examples/s]

Generating hmmt25_feb split:   0%|          | 0/48000 [00:00<?, ? examples/s]

In [3]:
import pandas as pd

light = ["task", "model_key", "data_id", "seed", "has_trigger", "finish_reason",
         "num_completion_tokens", "has_box", "is_correct"]
df = pd.concat([ds[s].select_columns(light).to_pandas() for s in ds], ignore_index=True)

# the grid is complete, every (task, model, question) pool holds exactly 80 seeds
print(df.shape, "| pool sizes", df.groupby(["task", "model_key", "data_id"]).size().unique())

# accuracy, mean is_correct over 30 questions x 80 seeds
acc = df.pivot_table(index="model_key", columns="task", values="is_correct", aggfunc="mean")
acc["mean"] = acc.mean(axis=1)
display(acc.sort_values("mean", ascending=False).round(3))

(192000, 9) | pool sizes [80]


task,aime24,aime25,brumo25,hmmt25_feb,mean
model_key,,,,,
Qwen3-30B-A3B-Thinking-2507,0.876,0.805,0.849,0.492,0.756
Qwen3-4B-Thinking-2507,0.773,0.732,0.748,0.410,0.666
Phi-4-reasoning-plus,0.755,0.683,0.724,0.460,0.656
gpt-oss-20b_high,0.753,0.700,0.659,0.448,0.640
gpt-oss-20b_medium,0.759,0.690,0.661,0.435,0.636
AceReason-Nemotron-1.1-7B,0.710,0.651,0.719,0.350,0.607
Phi-4-reasoning,0.706,0.601,0.703,0.389,0.600
OpenThinker2-32B,0.722,0.595,0.740,0.333,0.598
FuseO1-DeepSeekR1-QwQ-SkyT1-Flash-32B-Preview,0.728,0.585,0.715,0.313,0.585


## One model (4 datasets)

A per-model config is the full tier. Same 9,600 rows this model has in `meta`, plus
the six per-token lists. About 600 MB packed per model, so loading a model is preferable to looping over all 20.

In [4]:
model_name = "Phi-4-reasoning" # out of Models: ['meta', 'AceReason-Nemotron-1.1-7B', 'Bespoke-Stratos-7B', 'DeepSeek-R1-Distill-Qwen-1.5B', 'EXAONE-4.0-1.2B', 'FuseO1-DeepSeekR1-QwQ-SkyT1-Flash-32B-Preview', 'LIMO-v2', 'Light-R1-14B-DS', 'NVIDIA-Nemotron-Nano-9B-v2', 'OpenR1-Distill-7B', 'OpenReasoning-Nemotron-1.5B', 'OpenThinker2-32B', 'OpenThinker3-1.5B', 'Phi-4-reasoning', 'Phi-4-reasoning-plus', 'Qwen3-30B-A3B-Thinking-2507', 'Qwen3-4B-Thinking-2507', 'Sky-T1-32B-Flash', 'gpt-oss-20b_high', 'gpt-oss-20b_low', 'gpt-oss-20b_medium']

ds = load_dataset(repo_name, model_name)  # load all datasets for one model
# 4 (datasets) x 30 (questions) x 80 (runs) = 9,600 runs

ds

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

data/Phi-4-reasoning/aime24.parquet:   0%|          | 0.00/134M [00:00<?, ?B/s]

...

Generating aime24 split:   0%|          | 0/2400 [00:00<?, ? examples/s]

Generating aime25 split:   0%|          | 0/2400 [00:00<?, ? examples/s]

Generating brumo25 split:   0%|          | 0/2400 [00:00<?, ? examples/s]

Generating hmmt25_feb split:   0%|          | 0/2400 [00:00<?, ? examples/s]

DatasetDict({
    aime24: Dataset({
        features: ['task', 'model', 'model_key', 'data_id', 'seed', 'prompt', 'trigger', 'has_trigger', 'sampling', 'text', 'finish_reason', 'num_prompt_tokens', 'num_completion_tokens', 'ground_truth', 'ground_truth_accepted', 'answer_is_set', 'extracted_answer', 'extracted_answer_is_list', 'has_box', 'is_correct', 'prompt_logprob_sentinel', 'cv3b_label', 'cv3b_prob', 'cv3b_ctx_A', 'cv3b_ctx_B', 'cv3b_ctx_C', 'cv7b_label', 'cv7b_prob', 'cv7b_ctx_A', 'cv7b_ctx_B', 'cv7b_ctx_C', 'tokens'],
        num_rows: 2400
    })
    aime25: Dataset({
        features: ['task', 'model', 'model_key', 'data_id', 'seed', 'prompt', 'trigger', 'has_trigger', 'sampling', 'text', 'finish_reason', 'num_prompt_tokens', 'num_completion_tokens', 'ground_truth', 'ground_truth_accepted', 'answer_is_set', 'extracted_answer', 'extracted_answer_is_list', 'has_box', 'is_correct', 'prompt_logprob_sentinel', 'cv3b_label', 'cv3b_prob', 'cv3b_ctx_A', 'cv3b_ctx_B', 'cv3b_ctx_C', 'cv7

In [5]:
tk = ds["aime24"][0]["tokens"]

print("scalars   ", {k: round(v, 3) for k, v in tk.items() if not k.endswith("_list")})
print("list sizes", {k: len(v) for k, v in tk.items() if k.endswith("_list")})
print("first tokens", list(zip(tk["completion_token_list"][:8],
                               [round(x, 2) for x in tk["completion_logprob_list"][:8]],
                               tk["completion_rank_list"][:8])))

scalars    {'prompt_sum_logprob': -1130.272, 'prompt_avg_logprob': -2.921, 'prompt_ppl': 18.552, 'completion_sum_logprob': -308.523, 'completion_avg_logprob': -0.159, 'completion_ppl': 1.172}
list sizes {'prompt_token_list': 388, 'prompt_logprob_list': 388, 'prompt_rank_list': 388, 'completion_token_list': 1943, 'completion_logprob_list': 1943, 'completion_rank_list': 1943}
first tokens [('<think>', 0.0, 1), ('We', -0.01, 1), (' are', -0.05, 1), (' given', -0.26, 1), (' the', -1.93, 2), (' system', -0.39, 1), (' of', -1.12, 1), (' equations', -0.09, 1)]


## One dataset (20 models)

One split of `meta` puts all 20 models on the same 30 questions, 48,000 rows. This is
the view for leaderboards and for ranking work.

In [6]:
dataset_name = "aime24" # out of Datasets: ['aime24', 'aime25', 'brumo25', 'hmmt25_feb']

ds = load_dataset(repo_name, "meta", split=dataset_name)  
# 20 (models) x 30 (questions) x 80 (runs) = 48,000 runs


Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

In [7]:
records = ds.select_columns(["model_key", "data_id", "is_correct",
                             "num_completion_tokens"]).to_pandas()

leaderboard = records.groupby("model_key").agg(acc=("is_correct", "mean"),
                                               tokens=("num_completion_tokens", "mean"))
display(leaderboard.sort_values("acc", ascending=False).round(3))

by_question = records.groupby("data_id").is_correct.mean()
print("hardest questions", by_question.nsmallest(3).round(3).to_dict())
print("easiest questions", by_question.nlargest(3).round(3).to_dict())

,acc,tokens
model_key,,
Qwen3-30B-A3B-Thinking-2507,0.876,16160.179
Qwen3-4B-Thinking-2507,0.773,19480.379
gpt-oss-20b_medium,0.759,9403.574
Phi-4-reasoning-plus,0.755,14707.913
gpt-oss-20b_high,0.753,12226.815
Light-R1-14B-DS,0.734,11876.496
FuseO1-DeepSeekR1-QwQ-SkyT1-Flash-32B-Preview,0.728,10745.588
OpenThinker2-32B,0.722,10837.630
AceReason-Nemotron-1.1-7B,0.710,14304.960


hardest questions {5: 0.0, 17: 0.001, 6: 0.059}
easiest questions {2: 0.986, 0: 0.976, 8: 0.964}


## One dataset for a model

2,400 rows, 30 questions x 80 seeds, full tier. Rows are sorted by `data_id` and then
by `seed`, so question q sits in rows `q*80` to `q*80+79` and the seeds inside it run
1234 to 1313.

In [8]:
task = "aime25" # out of Datasets: ['aime24', 'aime25', 'brumo25', 'hmmt25_feb']
model_name = "Phi-4-reasoning" # out of Models: ['meta', 'AceReason-Nemotron-1.1-7B', 'Bespoke-Stratos-7B', 'DeepSeek-R1-Distill-Qwen-1.5B', 'EXAONE-4.0-1.2B', 'FuseO1-DeepSeekR1-QwQ-SkyT1-Flash-32B-Preview', 'LIMO-v2', 'Light-R1-14B-DS', 'NVIDIA-Nemotron-Nano-9B-v2', 'OpenR1-Distill-7B', 'OpenReasoning-Nemotron-1.5B', 'OpenThinker2-32B', 'OpenThinker3-1.5B', 'Phi-4-reasoning', 'Phi-4-reasoning-plus', 'Qwen3-30B-A3B-Thinking-2507', 'Qwen3-4B-Thinking-2507', 'Sky-T1-32B-Flash', 'gpt-oss-20b_high', 'gpt-oss-20b_low', 'gpt-oss-20b_medium']

ds = load_dataset(repo_name, model_name, split=task)  # 2400 rows = 30 questions * 80 attempts
# 30 (questions) x 80 (runs) = 2,400 runs


Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

In [9]:
records = ds.select_columns(["data_id", "is_correct", "num_completion_tokens", "finish_reason"]).to_pandas()

by_question = records.groupby("data_id").agg(acc=("is_correct", "mean"),
                                             tokens=("num_completion_tokens", "mean"),
                                             truncated=("finish_reason", lambda s: (s == "length").mean()))

print(f"{model_name} on {task}, accuracy {records.is_correct.mean():.3f}, "
      f"{(by_question.acc == 0).sum()} questions never solved, "
      f"{(by_question.acc == 1).sum()} always solved")
display(by_question.sort_values("acc").round(3))

Phi-4-reasoning on aime25, accuracy 0.601, 2 questions never solved, 5 always solved


,acc,tokens,truncated
data_id,,,
13,0.000,27492.512,0.512
14,0.000,23779.250,0.338
27,0.025,29862.975,0.575
12,0.038,26525.075,0.550
29,0.150,29753.675,0.712
21,0.238,4657.762,0.000
9,0.275,19550.875,0.262
28,0.388,18175.150,0.188
23,0.450,5612.888,0.000


## One candidate pool

The pool is the 80 traces of a single question. `pool[:n]` is a reproducible n-sample
budget, so a budget sweep is a row prefix and nothing needs re-running.

Use `select`. Rows are already grouped by question, so a pool comes back in 7 ms,
against 50 seconds for `filter`, which scans all 2,400 rows and their token lists.

In [10]:
question_id = 10
attempt_id = 0

pool = ds.select(range(question_id * 80, question_id * 80 + 80))
accepted = set(pool[0]["ground_truth_accepted"])

print(f"question {question_id}, seeds {pool[0]['seed']} to {pool[-1]['seed']}, accepted {sorted(accepted)}")
print(Counter(pool["extracted_answer"]).most_common(5))  # NotFound means no \boxed{} span

print(f"{'n':>3} {'mean acc':>9} {'vote':>10} {'vote ok':>8}")
for n in (1, 4, 8, 16, 32, 80):
    sub = pool[:n]
    votes = Counter(a for a in sub["extracted_answer"] if a != "NotFound")
    vote = votes.most_common(1)[0][0] if votes else "NotFound"
    print(f"{n:>3} {sum(sub['is_correct']) / n:>9.3f} {vote:>10} {str(vote in accepted):>8}")

question 10, seeds 1234 to 1313, accepted ['259']
[('259', 47), ('NotFound', 26), ('36', 3), ('267', 1), ('22', 1)]
  n  mean acc       vote  vote ok
  1     0.000        267    False
  4     0.250        267    False
  8     0.375        259     True
 16     0.438        259     True
 32     0.562        259     True
 80     0.588        259     True


## One trace

Everything the dataset knows about a single generation, including the per-token lists
that only exist in the per-model configs.

In [11]:
r = pool[attempt_id]

print(f"seed {r['seed']}, {r['num_completion_tokens']} tokens, finish {r['finish_reason']}, "
      f"answer {r['extracted_answer']}, correct {r['is_correct']}")
print("cv3b", r["cv3b_label"], round(r["cv3b_prob"], 4), "ctx", [round(r[f"cv3b_ctx_{c}"], 3) for c in "ABC"])
print("cv7b", r["cv7b_label"], round(r["cv7b_prob"], 4), "ctx", [round(r[f"cv7b_ctx_{c}"], 3) for c in "ABC"])

tk = r["tokens"]
lp, tok, rank = tk["completion_logprob_list"], tk["completion_token_list"], tk["completion_rank_list"]
print("avg logprob", round(tk["completion_avg_logprob"], 3), "ppl", round(tk["completion_ppl"], 3))
print("least confident", [(i, tok[i], round(lp[i], 2), rank[i])
                          for i in sorted(range(len(lp)), key=lambda i: lp[i])[:5]])

print(r["text"][:300], "...")

seed 1234, 20008 tokens, finish stop, answer 267, correct 0
cv3b B 0.9988 ctx [0.001, 0.922, 0.075]
cv7b B 0.9993 ctx [0.016, 0.973, 0.01]
avg logprob -0.155 ppl 1.168
least confident [(643, ' next', -4.0, 12), (255, ' period', -3.8, 13), (403, ' parameter', -3.56, 9), (1073, ' piece', -3.45, 10), (231, ' the', -3.21, 7)]
<think>We are given piecewise linear function f(x) defined by:
f(x) = { x if -1 ≤ x < 1, and 2 - x if 1 ≤ x < 3 } and f(x+4)=f(x) for all real numbers x. So it is periodic with period 4. The graph has sawtooth pattern. Then we have a parabola x = 34y^2. The intersection of the parabola and the graph ...
